# Lesson 10 — Images: anywhere text can go

Lesson 9 covered intent routing. This lesson covers images: sending a picture to a System One model in the `state`, inside a question, or as the options a question picks between. It also introduces **Box**, a question type that answers *where* something is in an image.

> **Dreamer-ready, not production-ready.** TypeSafe's own label. Images and Box are for prototypes. The request format, answers and limits can change between releases. They need the `TypeSafe-Danger-Zone-Images: true` header on every request that uses them.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/images (click to expand)</summary>

# Images

> Send images in a System One request: in the state, inside a question's instructions, and as the criteria a question is judged against.

> **Dreamer-ready.** Images and Box questions aren't yet up to our standards to call ready for production, but we're excited for dreamers and builders to play with them. Use them to build prototypes and try ideas. The request format, the answers, and the limits on this page can change between releases, and accuracy will change as we update the models. Tell us what you build, and what breaks, on [Discord](https://discord.gg/typesafe).
>
> To use images or Box questions, send the `TypeSafe-Danger-Zone-Images: true` header with each request that contains an image or a Box question.

An image can go anywhere text can go in a System One request: in the `state`, in a question's `instructions`, and in its `criteria`. Every question type accepts images, and the [Box](https://docs.typesafe.ai/primitives/box) question type returns where something is in an image.

## Turning on images

Images and [Box](https://docs.typesafe.ai/primitives/box) questions are off by default. To use them, send the `TypeSafe-Danger-Zone-Images: true` header with each request that contains an image or a Box question.

## Sending images in a request

Each image is listed once, in the request's top-level `assets` map. Its key in that map is a reference string. Write the reference wherever the image belongs, as a whole string value or inside a longer string, and the model receives the image in its place.

This request sends a photo of a found dog as the state, and puts the photo from a lost-dog poster directly into one question:

```json
{
  "model": "jev-latest",
  "state": {
    "photo": "a3f1c2e8-found"
  },
  "assets": {
    "a3f1c2e8-found": {"type": "image", "format": "jpg", "data_base64": "/9j/4AAQSkZJRg..."},
    "77b0d4a1-poster": {"type": "image", "format": "jpg", "data_base64": "/9j/4AAQSkZJRg..."}
  },
  "questions": {
    "same_dog": {
      "type": "noul",
      "instructions": "Is the dog in `photo` the same dog as in this picture: 77b0d4a1-poster?"
    },
    "has_collar": {
      "type": "noul",
      "instructions": "Is the dog in `photo` wearing a collar?"
    }
  }
}
```

Questions can refer to an image in the state by name or path in backticks, the same way they refer to any other field. An image that only one question uses goes directly in that question's `instructions` or `criteria`, as the poster photo does above.

Jev 1.13 and Jev 1.14 both accept images. [Box](https://docs.typesafe.ai/primitives/box) questions need Jev 1.14.

### Asset fields

`assets` (`map<string, Asset>`): the images in the request, keyed by reference string. Every reference must appear somewhere in the `state`, `instructions`, or `criteria`.

- `type` (`"image"`): the kind of asset. Optional. If you send it, it must be `"image"`.
- `part` (`string`): in a multipart request, the name of the file part that holds the image.
- `data_base64` (`string`): the image file, base64-encoded. Required unless `part` is set. Use the standard base64 alphabet with `=` padding, no line breaks or other whitespace, and no `data:` prefix.
- `format` (`"png" | "jpg" | "jpeg"`): the image's format, in lowercase. Required with `data_base64`. Optional with `part`. `"jpeg"` is another name for `"jpg"`.

Send each image one way, with `part` or `data_base64`. If an asset has both, the API uses `part`.

### References

References are matched as plain text anywhere in the request's strings, so a reference works mid-sentence as well as as a whole value. A short reference such as `cat` could also match inside `category`. Use references that can't occur in your text by accident. A new UUID for each image in each request is the safest choice.

* **Every asset must be referenced.** A request with an asset that is not referenced anywhere returns `400`.
* **Whole values match first.** A string that is exactly a reference becomes that image. In longer strings, references are replaced from left to right, and matches never overlap.
* **Each use adds the image again.** The file is uploaded once, but every place its reference appears adds another copy of the image to what the model reads, and every copy counts toward the expanded request limit.
* **Object keys are not replaced.** A Choice option name, a question ID, or any other object key stays text. To define a Choice option with a picture, put the reference in the option's description.
* **Answers contain the reference, not the image,** wherever they repeat your criteria, such as a Score's `legend`.
* **Keep keys short.** Each reference can be at most 1,024 bytes of UTF-8, and all references in a request together at most 65,536 bytes. A reference can't be empty or only whitespace.

## Uploading images

With the latest Python or JavaScript SDK, put an image object anywhere text can go. The SDK creates a reference for it and uploads the file. This example sends one doorbell photo as the state and asks two questions about it:

```typescript
import { openAsBlob } from "node:fs";
import { choice, image, noul, TypeSafeClient } from "@typesafe-ai/sdk";

const client = new TypeSafeClient();
const door = image(await openAsBlob("door-camera.jpg"));

const { answers } = await client.systemOne(
  {
    state: { door },
    questions: {
      person_present: noul("Is there a person in `door`?"),
      carrying: choice("What is the person in `door` carrying?", {
        package: "A box or parcel.",
        bag: "A bag of groceries or shopping.",
        nothing: "Empty hands.",
      }),
    },
  },
  { headers: { "TypeSafe-Danger-Zone-Images": "true" } },
);
console.log(answers);
```

## Image placement

**State.** Images in the `state` are part of the state, the same as any text or JSON there. Every question in the request is evaluated against them. A state can mix images with text, and can hold several images, labeled or nested like any other state data.

**Instructions.** An image in a question's `instructions` is part of that question and is sent only with that question. An image in `instructions` is sent to the model once for every question that includes it. When several questions use the same image, put it in the `state` once and refer to it by path from each question. That uses fewer tokens and less of the size limits.

**Criteria.** Choice options, Score levels, and Noul `true` and `false` definitions can each include images. An option can be an image on its own, a description with one or more examples, or text only.

## Limits

| Limit | Value |
| - | - |
| Format | PNG or JPEG, still images only. Animated PNGs are rejected. |
| File size | 16 MiB per image file. For base64, this is the size after decoding. |
| Width and height | 256 to 1568 pixels each. |
| Total pixels (width × height) | 78,400 to 2,458,624, the area of a 280 × 280 to a 1568 × 1568 square. A 256 × 256 image is too small. |
| Request body | 32 MiB, including the JSON, base64 text, and multipart framing. |
| All images in the request | 32 MiB, shared by every distinct image after the API processes it. |
| Expanded request | 32 MiB for the text, questions, structure, and images after every reference is replaced by its image. |

All limits include both ends of the range. An image outside them is rejected, never resized to fit. If any image in a request is rejected, the whole request fails: there are no partial answers.

## Preparing images

* **Resize before sending.** Scale the image so its longest side is at most 1568 pixels.
* **Rotate the pixels upright.** The API removes all metadata (EXIF, color profiles, comments) and does not apply the EXIF orientation.
* **Send a still image.** For an animation or a video, export the frame you want as a PNG or JPEG. Convert other formats, such as GIF, WebP, or HEIC, to PNG or JPEG.
* **Prefer standard JPEG.** A standard 8-bit baseline JPEG is the safest JPEG to send.

## Image tokens

Each image counts as input tokens, billed at the same per-token [input price](https://docs.typesafe.ai/models) as text. There is no separate charge for images. The `usage.input_tokens` field in the response includes the tokens for every image in the request.

## Errors

| Status | Cause |
| - | - |
| `400` | Images are not turned on: the request has an image or a Box question but no `TypeSafe-Danger-Zone-Images: true` header. |
| `400` | An image or asset was rejected: wrong format, damaged, animated, too small, unreferenced, invalid base64 or `format`, empty reference, or a broken multipart rule. |
| `400` | `max_tokens_exceeded`: the images and text together exceed the request's context length. |
| `413` | The request is over a size limit. |

</details>

## The core idea

Nothing about the questions changes. A `noul` is still a probability, a `choice` still picks one option. What's new is that some values in a request can be pictures.

In the SDK, wrap the file's bytes with `image()` and put the result anywhere text can go: in the `state`, in a question's `instructions`, or in its `criteria`. On the wire, the SDK does what the docs describe. It lists each image once in an `assets` map under a generated **reference** string, writes that reference wherever you placed the image, and uploads the file as its own part of a multipart request. The model sees the image wherever the reference appears.

Where you put it matters, because **each use adds the image again**. An image every question needs belongs in the `state`, once, with questions referring to it by name (`` `photo` ``), the way questions refer to any other field in the state. An image only one question needs, such as a reference photo to compare against, goes in that question's `instructions`.

## Images in the SDK

Images need `@typesafe-ai/sdk` 0.7.0 or newer. The [SDK README](https://github.com/typesafe-ai/typesafe-sdk-js#images) gives three ways to use an image, and you can combine them:

- **Place it in the content.** Put the `image()` value straight into `state`, a question's instructions or its criteria. This lesson does that.
- **Write it into text with `withAssets`.** Tag a template literal, as in `` withAssets`Does ${upload} match ${catalog}?` ``. An image in an *untagged* template literal is rejected before the request is sent.
- **Name it yourself in `assets`.** Pass `assets: { "<catalog>": image(...) }` and write `<catalog>` in plain strings, for example prompts stored in configuration.

Every request with an image or a Box question needs the `TypeSafe-Danger-Zone-Images: true` header. This lesson passes it per call. To send it on every request, set it in `TYPESAFE_HEADERS` (see the README).

In [ ]:
// Paths resolve from this notebook's folder, like the imports.
const photoBytes = await Deno.readFile("../../assets/images/bacon.jpg");

// Show the exact bytes the requests below send.
Deno.jupyter.image(photoBytes);

In [ ]:
import { choice, image, noul } from "npm:@typesafe-ai/sdk@^0.7.0";
import { client } from "../../assets/typesafe-client.ts";

const DANGER_ZONE = { headers: { "TypeSafe-Danger-Zone-Images": "true" } };

const photo = image(photoBytes);

const result = await client.systemOne(
  {
    state: { photo },
    questions: {
      image_is_cartoon: noul("Is the image in `photo` a cartoon or not?"),
      number_of_children: choice("How many children are in the image `photo`?", {
          none: "There are no children.",
          one: "There is one child.",
          two: "There are two children.",
          three: "There are three children.",
          four: "There are four children.",
          five: "There are five children.",
          six: "There are six children.",
          seven: "There are seven children.",
          eight: "There are eight children.",
          nine: "There are nine children.",
          ten: "There are ten children.",
          more_than_ten: "There are more than ten children.",
        }),
      period_of_image: choice("What period does the image `photo` depict?", {
          "1900s": "The image depicts the 1900s.",
          "1910s": "The image depicts the 1910s.",
          "1920s": "The image depicts the 1920s.",
          "1930s": "The image depicts the 1930s.",
          "1940s": "The image depicts the 1940s.",
          "1950s": "The image depicts the 1950s.",
          "1960s": "The image depicts the 1960s.",
          "1970s": "The image depicts the 1970s.",
          "1980s": "The image depicts the 1980s.",
          "1990s": "The image depicts the 1990s.",
          "2000s": "The image depicts the 2000s.",
          "2010s": "The image depicts the 2010s.",
          "2020s": "The image depicts the 2020s.",
        }),
      every_child_smiling: noul("Is every child in the image `photo` smiling?"),
    },
  },
  DANGER_ZONE,
);

console.log(JSON.stringify(result.answers, null, 2));
console.log("input tokens, image included:", result.usage.input_tokens);

## What just happened

`assets/images/bacon.jpg` is a 1280 × 853 illustration in a retro style: a children's birthday party, with a blindfolded boy pinning bacon onto a picture of a brain. Nothing in the request describes the picture. The model only gets the pixels.

Check the answers against the picture yourself. Count the children before you look at `number_of_children`, then read its `probabilities`, not just its `choice`. In a busy scene where children overlap, see whether the probability spreads across neighbouring counts.

The answers have exactly the shapes you've seen since lesson 2, typed by the SDK the same way. The model was given a picture instead of a sentence, and nothing downstream changed. That's the point: code that already thresholds a `noul` or switches on a `choice` works on images unchanged.

`usage.input_tokens` includes the image. Images are billed as input tokens at the text price, with no separate charge.

## Box: where is it?

Every question so far answers *what*. A **Box** question answers *where*. It returns the four edges of the smallest rectangle around the target, each as a fraction of the image's width (`left`, `right`) or height (`top`, `bottom`), measured from the top-left corner. Fractions mean the answer works at any image size: multiply by the pixel width or height to draw or crop.

From the [Box docs](https://docs.typesafe.ai/primitives/box): a Box question takes `instructions` saying what to find, and no `criteria`. It needs Jev 1.14, so send `jev-preview` as the `model`. In the SDK, ask one with `box()`. The answer has a `confidence`, which the SDK gives as `null` when the API doesn't report one. There's no recommended threshold yet, so tune it on your own images.

In [ ]:
import { box } from "npm:@typesafe-ai/sdk@^0.7.0";

const boxResult = await client.systemOne(
  {
    model: "jev-preview", // Box needs Jev 1.14
    state: { photo },
    questions: {
      red_balloon: box("the red balloon in `photo`"),
      girl_in_blue_dress: box("the girl in the blue dress in `photo`"),
      cake: box("the cake in `photo`"),
    },
  },
  DANGER_ZONE,
);

console.table(
  Object.fromEntries(
    Object.entries(boxResult.answers).map(([name, { box: edges, confidence }]) => [name, { ...edges, confidence }]),
  ),
);

In [ ]:
import { encodeBase64 } from "jsr:@std/encoding@^1/base64";

// bacon.jpg is 1280 × 853. Box edges are fractions, so scale them to pixels.
const WIDTH = 1280;
const HEIGHT = 853;

const rects = Object.entries(boxResult.answers).map(([name, { box: edges, confidence }]) => {
  const x = edges.left * WIDTH;
  const y = edges.top * HEIGHT;
  const w = (edges.right - edges.left) * WIDTH;
  const h = (edges.bottom - edges.top) * HEIGHT;
  // Right-align labels on the right half so they stay inside the image.
  const rightHalf = edges.left > 0.5;
  const label = confidence === null ? "no confidence" : `${Math.round(confidence * 100)}%`;
  return `<rect x="${x}" y="${y}" width="${w}" height="${h}" fill="none" stroke="#ff00aa" stroke-width="5"/>
  <text x="${rightHalf ? x + w : x}" y="${y + h + 30}" text-anchor="${rightHalf ? "end" : "start"}" fill="#ff00aa" stroke="white" stroke-width="1" font-size="28" font-family="sans-serif" font-weight="bold">${name} (${label})</text>`;
}).join("\n");

Deno.jupyter.svg`<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 ${WIDTH} ${HEIGHT}" width="100%">
  <image href="data:image/jpeg;base64,${encodeBase64(photoBytes)}" width="${WIDTH}" height="${HEIGHT}"/>
  ${rects}
</svg>`;

## Reading the boxes

Each box is drawn where the model says the object is, labelled with its `confidence`. Judge them by eye: there's no answer key for an illustration.

Then compare how right each box looks with its confidence. The Box docs suggest treating answers below your own threshold with more care, because there's no recommended threshold yet. When confidence is low, they suggest asking again on a larger image or a crop around the box, or padding the box before cropping.

## Try it yourself

Go back to the first request. Leave `photo` in the `state`, and also write the image into **two** of the questions' instructions with `withAssets`. Add `withAssets` to the SDK import, then write, for example, `` noul(withAssets`Is ${photo} a cartoon or not?`) ``.

The SDK still uploads the file once. Before you re-run it, predict: how many copies of the image does the model read now, and what happens to `usage.input_tokens`? The docs' rule is "each use adds the image again". Check whether the number matches your prediction, then put the request back. This is why an image several questions share belongs in the `state`, once.

## Before you send your own images

`bacon.jpg` already fits the limits. Your own photos may not. From the docs:

- **Size.** 256 to 1568 px on each side. Anything outside is rejected, never resized. Shrink a phone photo so its longest side is at most 1568 px.
- **Orientation.** The API strips EXIF metadata and doesn't apply EXIF rotation, so a photo that looks upright on your computer can arrive sideways. Rotate the pixels themselves.
- **Format.** PNG or JPEG stills only. Convert GIF, WebP or HEIC first.
- **All or nothing.** If one image in a request is rejected, the whole request fails, with no partial answers.

## Next up

The images docs list six patterns that build on this, including comparing against a reference picture, defining Choice options by example pictures, and spotting what changed between frames. They're all request shapes you've now seen. The question is where the image goes. See the "Patterns" table on [docs.typesafe.ai/images](https://docs.typesafe.ai/images#patterns).

For the State background, revisit `lessons/core-concepts/0001-what-is-a-system-one-model.ipynb` and `lessons/core-concepts/0005-structure.ipynb`.

Questions? Ask your agent. It has the same research this lesson was built from, in `research/typesafe-ai.md`.